In [1]:
import os
os.environ['SPARK_VERSION'] = '3.3'
os.environ["JAVA_HOME"] = '/usr/lib/jvm/java-8-openjdk-amd64/'

In [2]:
from pyspark.sql import SparkSession
import pydeequ

spark = SparkSession.builder \
    .appName("Loan Data ETL Pipeline") \
    .master("local[*]") \
    .config(
            "spark.jars.packages", "com.amazon.deequ:deequ:2.0.11-spark-3.3"
        )\
    .config("spark.jars.excludes", pydeequ.f2j_maven_coord)\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.fallback.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.memory.fraction", "0.2")
spark.conf.set("spark.sql.execution.arrow.pyspark.memory.max", "2g") 

25/06/17 11:04:20 WARN Utils: Your hostname, rohitkarki resolves to a loopback address: 127.0.1.1; using 10.13.164.166 instead (on interface wlp4s0)
25/06/17 11:04:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/rohitkarki/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/rohitkarki/.ivy2/cache
The jars for the packages stored in: /home/rohitkarki/.ivy2/jars
com.amazon.deequ#deequ added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9ce59b15-3541-430a-b68e-c2967fbfdcbc;1.0
	confs: [default]
	found com.amazon.deequ#deequ;2.0.11-spark-3.3 in central
	found org.scala-lang#scala-reflect;2.12.10 in central
	found org.scalanlp#breeze_2.12;1.2 in central
	found org.scalanlp#breeze-macros_2.12;1.2 in central
	found com.github.fommil.netlib#core;1.1.2 in central
	found net.sf.opencsv#opencsv;2.3 in central
	found com.github.wendykierp#JTransforms;3.1 in central
	found pl.edu.icm#JLargeArrays;1.5 in central
	found org.apache.commons#commons-math3;3.2 in central
	found com.chuusai#shapeless_2.12;2.3.3 in central
	found org.typelevel#macro-compat_2.12;1.1.1 in central
	found org.slf4j#slf4j-api;1.7.5 in central
	found org.typelevel#spire_2.12;0.17.0 in central
	found org.typelevel#spire-macros_2.12;

25/06/17 11:04:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/06/17 11:04:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
from pyspark.sql.functions import col, isnan, when, count, lit, split, to_date, hour, minute, second
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql.window import Window
from pyspark.sql import functions as F
# from pyspark.sql.functions import mode
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
import os
from datetime import datetime


In [9]:
schema = StructType([
    StructField("Loan_id", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("Married", StringType(), True),
    StructField("Dependents", IntegerType(), True),
    StructField("Education", StringType(), True),
    StructField("Self_Employed", StringType(), True),
    StructField("ApplicantIncome", IntegerType(), True),
    StructField("CoapplicantIncome", IntegerType(), True),
    StructField("LoanAmount", IntegerType(), True),
    StructField("Loan_Amount_Term", IntegerType(), True),
    StructField("Credit_History", IntegerType(), True),
    StructField("Property_Area", StringType(), True),
    StructField("Loan_Status", StringType(), True),
])

In [10]:
try:
    input_path = "hdfs://localhost:9000/user/hive/warehouse/loan.csv"
    # Read the CSV file with the specified schema
    raw_df = spark.read.csv(input_path, header=True, schema=schema)

    print("CSV file read successfully.")
    print(f"Sample data")
    raw_df.show(5, truncate=False)

    # print("Initial data statistics:")
    # raw_df.describe().show()
    
    null_counts = raw_df.select([count(when(col(c).isNull() | isnan(col(c)), c)).alias(c) for c in raw_df.columns])
    null_counts.show(truncate=False)
    # null_counts.show()
except Exception as e:
    print(f"Error reading CSV file: {e}")

CSV file read successfully.
Sample data


+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|Loan_id |Gender|Married|Dependents|Education   |Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|LP001002|Male  |No     |0         |Graduate    |No           |5849           |0                |null      |360             |1             |Urban        |Y          |
|LP001003|Male  |Yes    |1         |Graduate    |No           |4583           |1508             |128       |360             |1             |Rural        |N          |
|LP001005|Male  |Yes    |0         |Graduate    |Yes          |3000           |0                |66        |360             |1             |Urban        |Y          

In [ ]:
import pydeequ
from pydeequ.analyzers import AnalysisRunner, Size, Completeness, ApproxCountDistinct, Mean, AnalyzerContext

In [12]:
analysisResult = AnalysisRunner(spark) \
                    .onData(raw_df) \
                    .addAnalyzer(Size()) \
                    .addAnalyzer(Completeness("Married")) \
                    .addAnalyzer(ApproxCountDistinct("Loan_id")) \
                    .addAnalyzer(Mean("LoanAmount")) \
                    .run()
                    # .addAnalyzer(Compliance("top star_rating", "star_rating >= 4.0")) \
                    # .addAnalyzer(Correlation("total_votes", "star_rating")) \
                    # .addAnalyzer(Correlation("total_votes", "helpful_votes")) \
                    
analysisResult_df = AnalyzerContext.successMetricsAsDataFrame(spark, analysisResult)
analysisResult_df.show()

+-------+----------+-------------------+------------------+
| entity|  instance|               name|             value|
+-------+----------+-------------------+------------------+
|Dataset|         *|               Size|             614.0|
| Column|   Married|       Completeness| 0.995114006514658|
| Column|   Loan_id|ApproxCountDistinct|             607.0|
| Column|LoanAmount|               Mean|146.41216216216216|
+-------+----------+-------------------+------------------+



/home/rohitkarki/.local/lib/python3.10/site-packages/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [ ]:
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite

In [ ]:
check = Check(spark, CheckLevel.Warning, "Loan Dataset Review Check")
checkResult = VerificationSuite(spark)\
    .onData(raw_df)\
    .addAnomalyCheck